# Text Features: Extracting Signals from Text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/07_text_features.ipynb)

## Objectives
- Extract statistical features from text
- Understand text preprocessing techniques
- Implement word/document frequency features
- Learn vectorization methods (TF-IDF, Bag of Words)
- Handle unstructured text data effectively

print("""📝 TEXT FEATURES:

1. LENGTH FEATURES:
   • Character count
   • Word count
   • Sentence count
   • Average word length
   • Average sentence length

2. PUNCTUATION & CASE:
   • Number of punctuation marks
   • Number of uppercase letters
   • Number of digits
   • Number of special characters

3. LINGUISTIC:
   • Number of unique words
   • Lexical diversity (unique/total)
   • Stop word ratio
   • Sentiment (positive/negative words)

4. BAG-OF-WORDS:
   • Word frequency
   • Term frequency (TF)

5. ADVANCED:
   • TF-IDF (Term Frequency-Inverse Document Frequency)
   • Word embeddings (Word2Vec, GloVe)
   • BERT embeddings""")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

# Product reviews dataset
reviews = [
    "This product is amazing! It works perfectly. Highly recommend!",
    "Terrible quality. Broke after one week.",
    "Good product. Average price. Worth buying.",
    "Excellent! Exceeded all expectations. Best purchase ever!",
    "Not satisfied. Many defects found. Do not buy!!!",
    "It's okay, nothing special.",
    "Fantastic quality at reasonable price. Love it!",
    "Waste of money. Disappointed.",
    "Perfect product, fast shipping, great service!",
    "Average. Could be better for the price."
]

ratings = [5, 1, 3, 5, 1, 3, 4, 1, 5, 3]

df = pd.DataFrame({
    'review': reviews,
    'rating': ratings
})

print(f"Dataset shape: {df.shape}")
print(f"\nSample reviews:")
print(df.head(5))

# Character-level features
df['char_count'] = df['review'].str.len()
df['word_count'] = df['review'].str.split().str.len()
df['sentence_count'] = df['review'].str.split(r'[.!?]').str.len() - 1  # -1 for last empty split
df['avg_word_length'] = df['char_count'] / df['word_count']
df['avg_sentence_length'] = df['word_count'] / (df['sentence_count'] + 1)  # +1 to avoid division by zero

print("📊 Length Features:")
print(df[['review', 'char_count', 'word_count', 'sentence_count', 'avg_word_length']].head(10))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].scatter(df['word_count'], df['rating'], alpha=0.6, s=100)
axes[0, 0].set_xlabel('Word Count')
axes[0, 0].set_ylabel('Rating')
axes[0, 0].set_title('Word Count vs Rating')

axes[0, 1].hist(df['word_count'], bins=5, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_title('Word Count Distribution')

axes[1, 0].hist(df['avg_word_length'], bins=5, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Average Word Length')
axes[1, 0].set_title('Average Word Length Distribution')

axes[1, 1].scatter(df['char_count'], df['rating'], alpha=0.6, s=100, color='red')
axes[1, 1].set_xlabel('Character Count')
axes[1, 1].set_ylabel('Rating')
axes[1, 1].set_title('Character Count vs Rating')

plt.tight_layout()
plt.show()

# Punctuation and special characters
df['punct_count'] = df['review'].str.findall(r'[!?.,;:]').str.len()
df['exclamation_count'] = df['review'].str.count('!')
df['question_count'] = df['review'].str.count('\?')
df['uppercase_count'] = df['review'].str.count('[A-Z]')
df['digit_count'] = df['review'].str.count('[0-9]')

print("📊 Punctuation & Case Features:")
print(df[['review', 'punct_count', 'exclamation_count', 'uppercase_count']].head(10))

print(f"\nCorrelation with rating:")
punct_features = ['punct_count', 'exclamation_count', 'question_count', 'uppercase_count']
for feat in punct_features:
    corr = df[feat].corr(df['rating'])
    print(f"  {feat}: {corr:.3f}")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def preprocess_text(text):
    """Clean and standardize text"""
    # Lowercase
    text = text.lower()
    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

df['review_clean'] = df['review'].apply(preprocess_text)

print("📊 Text Preprocessing:")
print(f"Original: {df['review'].iloc[0]}")
print(f"Cleaned:  {df['review_clean'].iloc[0]}")

print(f"\nAll cleaned reviews:")
for i, clean in enumerate(df['review_clean']):
    print(f"{i+1}. {clean}")

stop_words = set(stopwords.words('english'))

def count_stopwords(text):
    words = text.lower().split()
    return sum(1 for word in words if word in stop_words)

df['stopword_count'] = df['review'].apply(count_stopwords)
df['stopword_ratio'] = df['stopword_count'] / (df['word_count'] + 1)

print("📊 Stop Word Features:")
print(df[['review', 'stopword_count', 'stopword_ratio', 'rating']].head(10))

print(f"\nStop word correlation with rating:")
print(f"  Correlation: {df['stopword_count'].corr(df['rating']):.3f}")

# Count vectorizer
vectorizer = CountVectorizer(max_features=10, lowercase=True)
bow_matrix = vectorizer.fit_transform(df['review_clean'])
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())

print("📊 Bag of Words (Word Frequency):")
print(f"\nFeatures (top 10 words): {list(vectorizer.get_feature_names_out())}")
print(f"\nBag of Words matrix (First 5 samples):")
print(bow_df.head())

# Word frequency across all reviews
word_freq = bow_df.sum().sort_values(ascending=False)
print(f"\nWord frequencies:")
print(word_freq)

# Visualization
plt.figure(figsize=(12, 5))
word_freq.plot(kind='bar', color='steelblue')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.title('Word Frequency in Reviews (Bag of Words)')
plt.tight_layout()
plt.show()

# TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=10, lowercase=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['review_clean'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print("📊 TF-IDF Vectorization:")
print(f"\nTF-IDF matrix (First 5 samples):")
print(tfidf_df.head())

print(f"\nTF-IDF scores (average across reviews):")
print(tfidf_df.mean().sort_values(ascending=False))

# Compare BOW vs TF-IDF
comparison = pd.DataFrame({
    'Word': tfidf_vectorizer.get_feature_names_out(),
    'Frequency': bow_df.sum().values,
    'TF-IDF': tfidf_df.mean().values
}).sort_values('TF-IDF', ascending=False)

print(f"\nComparison - Frequency vs TF-IDF:")
print(comparison)

# Simple sentiment word lists
positive_words = {'amazing', 'excellent', 'fantastic', 'good', 'great', 'perfect', 'love', 'best'}
negative_words = {'terrible', 'bad', 'awful', 'hate', 'waste', 'defects', 'disappointed', 'poor'}

def count_sentiment(text, word_list):
    words = set(text.lower().split())
    return sum(1 for word in words if word in word_list)

df['positive_words'] = df['review'].apply(lambda x: count_sentiment(x, positive_words))
df['negative_words'] = df['review'].apply(lambda x: count_sentiment(x, negative_words))
df['sentiment_score'] = df['positive_words'] - df['negative_words']

print("📊 Sentiment Features:")
print(df[['review', 'positive_words', 'negative_words', 'sentiment_score', 'rating']].head(10))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['sentiment_score'], df['rating'], s=100, alpha=0.6)
axes[0].set_xlabel('Sentiment Score')
axes[0].set_ylabel('Rating')
axes[0].set_title('Sentiment Score vs Rating')
axes[0].grid(True, alpha=0.3)

sentiment_rating = df.groupby('sentiment_score')['rating'].mean()
axes[1].bar(sentiment_rating.index, sentiment_rating.values, color='green', alpha=0.7)
axes[1].set_xlabel('Sentiment Score')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('Average Rating by Sentiment Score')

plt.tight_layout()
plt.show()

print(f"\nSentiment-Rating Correlation: {df['sentiment_score'].corr(df['rating']):.3f}")

def extract_text_features(df, text_col, target_col=None):
    """Extract comprehensive text features"""
    df = df.copy()
    text = df[text_col]
    
    # Length features
    df['text_char_count'] = text.str.len()
    df['text_word_count'] = text.str.split().str.len()
    df['text_sentence_count'] = text.str.count(r'[.!?]')
    df['text_avg_word_length'] = df['text_char_count'] / df['text_word_count']
    
    # Punctuation
    df['text_exclamation'] = text.str.count('!')
    df['text_question'] = text.str.count('\?')
    df['text_uppercase'] = text.str.count('[A-Z]')
    
    # Stop words
    df['text_stopword_ratio'] = text.apply(lambda x: count_stopwords(x) / (len(x.split()) + 1))
    
    # Sentiment
    df['text_positive_words'] = text.apply(lambda x: count_sentiment(x, positive_words))
    df['text_negative_words'] = text.apply(lambda x: count_sentiment(x, negative_words))
    df['text_sentiment_score'] = df['text_positive_words'] - df['text_negative_words']
    
    return df

# Apply pipeline
df_features = extract_text_features(df, 'review')

print(f"✅ Features before: {len(df.columns)}")
print(f"✅ Features after: {len(df_features.columns)}")
print(f"✅ New text features: {len(df_features.columns) - len(df.columns)}")

print(f"\nText feature columns:")
text_cols = [col for col in df_features.columns if 'text_' in col]
for col in text_cols:
    print(f"  • {col}")

print("""\n🎯 TEXT FEATURE ENGINEERING BEST PRACTICES:

✅ DO:
   1. Start with simple features (length, punctuation)
   2. Create domain-specific lexicons (positive/negative words)
   3. Remove noise (special chars, URLs, emails)
   4. Standardize case and whitespace
   5. Combine statistical + semantic features
   6. Store feature importance/correlations

❌ DON'T:
   1. Use raw TF-IDF without domain context
   2. Trust automatic tokenization without review
   3. Mix languages without handling
   4. Forget to handle missing/empty text
   5. Extract too many features (curse of dimensionality)

⚠️  COMMON ISSUES:
   • Stop words: Remove or weight differently?
   • Stemming/Lemmatization: May lose meaning
   • Vocabulary size: Thousands of features
   • Sparse data: Most TF-IDF values are 0

🔧 SOLUTIONS:
   • Use max_features in vectorizers
   • Apply min/max document frequency filters
   • Combine statistical + learned features
   • Use dimensionality reduction (PCA, SVD)
""")

print("""\n📚 KEY TAKEAWAYS:

Text Feature Types:
✓ Simple: Length, punctuation, case
✓ Frequency: Bag-of-Words, TF-IDF
✓ Semantic: Sentiment, domain lexicons
✓ Advanced: Embeddings (Word2Vec, BERT)

Feature Creation Strategy:
1. Start simple (char/word count)
2. Add linguistic features
3. Create domain dictionary
4. Use vectorizers (BOW/TF-IDF)
5. Select top features

Trade-offs:
• Simple features: Interpretable, may miss patterns
• Complex features: Better predictions, less interpretable
• Find balance for your use case

Feature Engineering Complete!
Next: Select features, scale, model!""")